<a href="https://colab.research.google.com/github/janithars/gvfa-streamlit-dashboard/blob/main/Interactive_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 82.2 MB/s eta 0:00:00


In [ ]:
!rm -rf visual_outputs
!mkdir -p visual_outputs

In [3]:
!python experiment_full_4variants.py

Using device: cpu
/content/experiment_full_4variants.py:51: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be removed in SciPy 2.0.0.
  objects.append(pkl.load(f, encoding="latin1") if sys.version_info > (3, 0) else pkl.load(f))

GVFA configuration: PHI1
/content/experiment_full_4variants.py:122: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(indices, values, torch.Size(mx.shape), device=device).coalesce()
Node embedding shape: (2708, 30000)
Variant 1 - Raw similarity | AUC: 0.8103, AP: 0.8186, HR@100: 0.9300
Variant 2 -

In [6]:
%%writefile app.py
import os
import pickle
import traceback

import numpy as np
import pandas as pd
import streamlit as st


# -----------------------------
# Page setup
# -----------------------------
st.set_page_config(
    page_title="GVFA LP Interactive Dashboard",
    page_icon="📊",
    layout="wide",
)

st.title("📊 GVFA Link Prediction Interactive Dashboard")
st.caption("Clean Streamlit app for loading and inspecting GVFA visualization outputs.")


# -----------------------------
# Helpers
# -----------------------------
PKL_PATHS = [
    "visual_outputs/gvfa_visual_phi4.pkl",
    "gvfa_visual_phi4.pkl",
]


def find_pickle_file():
    for path in PKL_PATHS:
        if os.path.exists(path):
            return path

    # fallback search
    for root, _, files in os.walk("."):
        for file in files:
            if file.endswith(".pkl") and "gvfa" in file.lower():
                return os.path.join(root, file)

    return None


@st.cache_data(show_spinner=True)
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def describe_object(obj):
    """Return basic metadata about a Python object."""
    info = {
        "type": type(obj).__name__,
    }

    if isinstance(obj, dict):
        info["keys"] = list(obj.keys())
        info["num_keys"] = len(obj)

    elif isinstance(obj, (list, tuple)):
        info["length"] = len(obj)
        if len(obj) > 0:
            info["first_item_type"] = type(obj[0]).__name__

    elif isinstance(obj, np.ndarray):
        info["shape"] = obj.shape
        info["dtype"] = str(obj.dtype)

    elif isinstance(obj, pd.DataFrame):
        info["shape"] = obj.shape
        info["columns"] = list(obj.columns)

    return info


def safe_dataframe(x, max_rows=5000):
    """Convert common structures to a displayable dataframe."""
    if isinstance(x, pd.DataFrame):
        return x.head(max_rows)

    if isinstance(x, pd.Series):
        return x.head(max_rows).to_frame()

    if isinstance(x, np.ndarray):
        arr = x
        if arr.ndim == 1:
            return pd.DataFrame({"value": arr[:max_rows]})
        if arr.ndim == 2:
            rows = min(max_rows, arr.shape[0])
            cols = min(100, arr.shape[1])
            return pd.DataFrame(arr[:rows, :cols])
        return pd.DataFrame({"summary": [f"ndarray with shape {arr.shape}"]})

    if isinstance(x, list):
        if len(x) == 0:
            return pd.DataFrame({"value": []})
        if isinstance(x[0], dict):
            return pd.DataFrame(x).head(max_rows)
        return pd.DataFrame({"value": x[:max_rows]})

    if isinstance(x, dict):
        rows = []
        for k, v in x.items():
            rows.append(
                {
                    "key": str(k),
                    "type": type(v).__name__,
                    "shape_or_len": getattr(v, "shape", len(v) if hasattr(v, "__len__") else ""),
                    "preview": str(v)[:200],
                }
            )
        return pd.DataFrame(rows)

    return pd.DataFrame({"value": [str(x)[:1000]]})


def try_get_metric(data, possible_keys):
    if not isinstance(data, dict):
        return None
    for key in possible_keys:
        if key in data:
            return data[key]
    return None


# -----------------------------
# Sidebar
# -----------------------------
st.sidebar.header("⚙️ Controls")

pkl_path = find_pickle_file()

if pkl_path is None:
    st.error("No GVFA `.pkl` file found.")
    st.info(
        "Expected file like `visual_outputs/gvfa_visual_phi4.pkl`. "
        "Please run your experiment script first."
    )

    st.code(
        """
!rm -rf visual_outputs
!mkdir -p visual_outputs
!python experiment_full_4variants.py
        """,
        language="python",
    )
    st.stop()

st.sidebar.success(f"Found pickle file: {pkl_path}")

if st.sidebar.button("Clear Streamlit cache"):
    st.cache_data.clear()
    st.sidebar.info("Cache cleared. Refresh the page.")


# -----------------------------
# Load data
# -----------------------------
try:
    with st.spinner(f"Loading {pkl_path} ..."):
        data = load_pickle(pkl_path)
except Exception:
    st.error("Failed to load pickle file.")
    st.code(traceback.format_exc(), language="python")
    st.stop()


# -----------------------------
# Top summary
# -----------------------------
st.subheader("1. Loaded Object Summary")

summary = describe_object(data)

col1, col2, col3 = st.columns(3)
col1.metric("Object type", summary.get("type", "unknown"))
col2.metric("File", os.path.basename(pkl_path))
col3.metric("File size MB", f"{os.path.getsize(pkl_path) / (1024 * 1024):.2f}")

st.json(summary)


# -----------------------------
# If dictionary, inspect keys
# -----------------------------
if isinstance(data, dict):
    st.subheader("2. Available Data Keys")

    key_rows = []
    for key, value in data.items():
        key_rows.append(
            {
                "key": key,
                "type": type(value).__name__,
                "shape": str(getattr(value, "shape", "")),
                "length": len(value) if hasattr(value, "__len__") else "",
                "preview": str(value)[:150],
            }
        )

    key_df = pd.DataFrame(key_rows)
    st.dataframe(key_df, use_container_width=True)

    selected_key = st.sidebar.selectbox("Select data key to inspect", list(data.keys()))

    st.subheader(f"3. Inspect Key: `{selected_key}`")
    selected_value = data[selected_key]

    st.write("Type:", type(selected_value).__name__)
    if hasattr(selected_value, "shape"):
        st.write("Shape:", selected_value.shape)

    st.dataframe(safe_dataframe(selected_value), use_container_width=True)

else:
    st.subheader("2. Data Preview")
    st.dataframe(safe_dataframe(data), use_container_width=True)


# -----------------------------
# Metrics / AUC section
# -----------------------------
st.subheader("4. Metrics / Scores")

if isinstance(data, dict):
    possible_metric_keys = [
        "results",
        "metrics",
        "final_results",
        "auc",
        "aucs",
        "scores",
        "variant_results",
    ]

    metric_obj = try_get_metric(data, possible_metric_keys)

    if metric_obj is not None:
        st.write("Detected metric object:")
        st.dataframe(safe_dataframe(metric_obj), use_container_width=True)
    else:
        st.info(
            "No obvious metric key found. Inspect the keys above and select the relevant one from the sidebar."
        )
else:
    st.info("Loaded object is not a dictionary, so automatic metric detection was skipped.")


# -----------------------------
# Matrix visualization
# -----------------------------
st.subheader("5. Matrix / Array Visualization")

array_candidates = {}

if isinstance(data, dict):
    for key, value in data.items():
        if isinstance(value, np.ndarray) and value.ndim == 2:
            array_candidates[key] = value
        elif isinstance(value, pd.DataFrame):
            numeric_df = value.select_dtypes(include=[np.number])
            if numeric_df.shape[0] > 0 and numeric_df.shape[1] > 0:
                array_candidates[key] = numeric_df.values

if array_candidates:
    arr_key = st.sidebar.selectbox("Select matrix to visualize", list(array_candidates.keys()))
    arr = array_candidates[arr_key]

    max_n = min(arr.shape[0], arr.shape[1], 300)
    n = st.sidebar.slider("Matrix display size", min_value=20, max_value=max_n, value=min(100, max_n))

    st.write(f"Showing top-left `{n} x {n}` block from `{arr_key}` with original shape `{arr.shape}`.")

    matrix_df = pd.DataFrame(arr[:n, :n])
    st.dataframe(matrix_df, use_container_width=True)

    st.write("Heatmap")
    st.line_chart(pd.DataFrame(arr[: min(500, arr.shape[0]), 0]))
else:
    st.info("No 2D NumPy array or numeric dataframe found for matrix visualization.")


# -----------------------------
# Raw debug section
# -----------------------------
with st.expander("🔎 Raw Debug Info"):
    st.write("Pickle path:", pkl_path)
    st.write("Current directory:", os.getcwd())
    st.write("Directory files:", os.listdir("."))
    if os.path.exists("visual_outputs"):
        st.write("visual_outputs files:", os.listdir("visual_outputs"))

st.success("Dashboard loaded successfully.")

Overwriting app.py


In [24]:
!pkill -f streamlit || true
!rm -f log.txt

^C


In [25]:


%%writefile ~/.streamlit/config.toml
[server]
headless = true
address = "0.0.0.0"
port = 8501
enableCORS = false
enableXsrfProtection = false

[browser]
gatherUsageStats = false

Overwriting /root/.streamlit/config.toml


In [26]:
!mkdir -p ~/.streamlit

In [27]:
!streamlit run app.py > log.txt 2>&1 &

In [28]:
!tail -n 80 log.txt

2026-05-28 12:07:03.377 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.48.233.114:8501



In [29]:
from google.colab import output
print(output.eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-m-s-kkb-use4a2-2tu584ccy6wxc-a.us-east4-2.prod.colab.dev


Check

In [30]:
!ls -lah
!ls -lah visual_outputs

total 88K
drwxr-xr-x 1 root root 4.0K May 28 12:07 .
drwxr-xr-x 1 root root 4.0K May 28 04:15 ..
-rw-r--r-- 1 root root  31K May 28 12:06 app.py
drwxr-xr-x 4 root root 4.0K May 26 13:25 .config
drwxr-xr-x 2 root root 4.0K May 28 04:28 data
-rw-r--r-- 1 root root  20K May 28 04:26 experiment_full_4variants.py
drwxr-xr-x 2 root root 4.0K May 28 12:06 .ipynb_checkpoints
-rw-r--r-- 1 root root 3.4K May 28 12:08 log.txt
-rw-r--r-- 1 root root   64 May 28 04:26 requirements.txt
drwxr-xr-x 1 root root 4.0K May 26 13:25 sample_data
drwxr-xr-x 2 root root 4.0K May 28 11:16 visual_outputs
total 453M
drwxr-xr-x 2 root root 4.0K May 28 11:16 .
drwxr-xr-x 1 root root 4.0K May 28 12:07 ..
-rw-r--r-- 1 root root 114M May 28 06:09 gvfa_visual_phi1.pkl
-rw-r--r-- 1 root root 114M May 28 07:50 gvfa_visual_phi2.pkl
-rw-r--r-- 1 root root 114M May 28 09:34 gvfa_visual_phi3.pkl
-rw-r--r-- 1 root root 114M May 28 11:16 gvfa_visual_phi4.pkl


In [31]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [32]:
!mkdir -p /content/drive/MyDrive/GVFA_Meeting_Project

In [33]:
!cp app.py /content/drive/MyDrive/GVFA_Meeting_Project/
!cp experiment_full_4variants.py /content/drive/MyDrive/GVFA_Meeting_Project/
!cp requirements.txt /content/drive/MyDrive/GVFA_Meeting_Project/
!cp -r visual_outputs /content/drive/MyDrive/GVFA_Meeting_Project/

In [34]:
!zip -r GVFA_Meeting_Project.zip app.py experiment_full_4variants.py requirements.txt visual_outputs

  adding: app.py (deflated 75%)
  adding: experiment_full_4variants.py (deflated 72%)
  adding: requirements.txt (deflated 9%)
  adding: visual_outputs/ (stored 0%)
  adding: visual_outputs/gvfa_visual_phi4.pkl (deflated 31%)
  adding: visual_outputs/gvfa_visual_phi2.pkl (deflated 31%)
  adding: visual_outputs/gvfa_visual_phi1.pkl (deflated 26%)
  adding: visual_outputs/gvfa_visual_phi3.pkl (deflated 27%)


In [35]:
from google.colab import files
files.download("GVFA_Meeting_Project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>